# 27 — Streaming RAG

> **Tier 7 | Production**

## What is Streaming RAG?

All previous notebooks used `invoke_model()` — a blocking call that waits for the full
response before returning. In a production interface the user sees nothing until the
entire answer is ready, which feels slow even when total latency is acceptable.

**Streaming RAG** uses Bedrock's `invoke_model_with_response_stream()` API: tokens
are yielded as the model generates them, so the first word appears within ~500ms
regardless of answer length.

### Key metrics

| Metric | Definition |
|--------|------------|
| **TTFT** | Time-To-First-Token — latency until the first chunk arrives |
| **Total latency** | Time until the full response is complete |
| **Tokens/s** | Generation throughput once streaming starts |

Streaming does not reduce total latency — it reduces *perceived* latency by surfacing
partial output immediately.

## PDF Framework: pypdf `extract_text(extraction_mode="layout")`

NB 23 used `pypdf` with plain `page.extract_text()`. This notebook uses the
`extraction_mode="layout"` parameter introduced in **pypdf ≥ 3.17**, which preserves
horizontal spacing and column alignment — producing cleaner chunks for multi-column PDFs.

| pypdf mode | Behaviour |
|-----------|----------|
| `extract_text()` | Plain concatenation (NB 23) |
| `extract_text(extraction_mode="layout")` | Preserves whitespace layout (this notebook) |

## Flow Diagram


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pypdf layout mode"]
        PDF(["📄 climate.pdf"])
        PDF --> PY["pypdf PdfReader\nextract_text(mode=layout)"]
        PY --> CH["Text chunks"]
        CH --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\nstreaming_rag_27")]
    end
    subgraph STREAM ["⚡  STREAMING RAG"]
        Q(["❓ Query"])
        Q --> RET["Vector Search\ntop-K chunks"]
        RET --> QDRANT
        QDRANT --> CTX["Build prompt"]
        CTX --> STREAM2["invoke_model_with_response_stream"]
        STREAM2 -->|token chunk| T1["token₁"]
        STREAM2 -->|token chunk| T2["token₂"]
        STREAM2 -->|token chunk| TN["...tokenₙ"]
        T1 --> OUT(["🖨 Streamed output"])
        T2 --> OUT
        TN --> OUT
    end
    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style STREAM fill:#fdf4ff,stroke:#9333ea,color:#3b0764
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "pypdf", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid
from typing import List, Dict, Generator
from dotenv import load_dotenv

import boto3
import pypdf
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print("Imports OK")
print(f"pypdf version: {pypdf.__version__}")


## Step 2 — Configuration

In [ ]:
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")

COLLECTION_NAME = "streaming_rag_27"
EMBEDDING_DIM   = 1024
CHUNK_SIZE      = 500
CHUNK_OVERLAP   = 50
TOP_K           = 5

PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection : {COLLECTION_NAME}")
print(f"PDF        : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")


## Step 3 — Qdrant Setup

In [ ]:
def make_qdrant(url='', api_key=''):
    if url:
        try:
            kw = {'url': url}
            if api_key: kw['api_key'] = api_key
            client = QdrantClient(**kw)
            client.get_collections()
            print(f'Qdrant Cloud: {url}')
            return client
        except Exception as e:
            print(f'Qdrant Cloud unavailable ({e}), falling back...')
    print('Using Qdrant in-memory')
    return QdrantClient(':memory:')


qdrant = make_qdrant(QDRANT_URL, QDRANT_API_KEY)

if COLLECTION_NAME in [c.name for c in qdrant.get_collections().collections]:
    qdrant.delete_collection(COLLECTION_NAME)
qdrant.create_collection(
    COLLECTION_NAME,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
print(f'Created "{COLLECTION_NAME}" (dim={EMBEDDING_DIM})')


## Step 4 — Bedrock Helpers (embed + blocking LLM)

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

def call_llm_blocking(system: str, user_content: str, max_tokens: int = 1024) -> str:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system,
        "messages": [{"role": "user", "content": user_content}],
    })
    resp = bedrock_rt.invoke_model(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["content"][0]["text"]

test_emb = embed_text("streaming rag pypdf layout mode test")
print(f"Embedding OK — dim={len(test_emb)}")


## Step 5 — PDF Loading with pypdf `extraction_mode="layout"`

The `layout` extraction mode (pypdf ≥ 3.17) inserts spaces to preserve the horizontal
position of words, producing cleaner text from multi-column or spaced-layout PDFs.
We normalise the extra whitespace before chunking.


In [ ]:
def recursive_split(text: str, size: int, overlap: int) -> List[str]:
    if len(text) <= size:
        return [text] if text.strip() else []
    chunks, start = [], 0
    while start < len(text):
        chunk = text[start:start + size].strip()
        if chunk:
            chunks.append(chunk)
        start += size - overlap
    return chunks


def load_pdf_pypdf_layout(path: str):
    reader  = pypdf.PdfReader(path)
    n_pages = len(reader.pages)
    chunks  = []

    for page_num, page in enumerate(reader.pages):
        try:
            raw = page.extract_text(extraction_mode="layout") or ""
        except TypeError:
            # fallback for older pypdf that doesn't support extraction_mode
            raw = page.extract_text() or ""
        text = ' '.join(raw.split()).strip()
        if not text:
            continue
        for sub in recursive_split(text, CHUNK_SIZE, CHUNK_OVERLAP):
            chunks.append({
                'text'      : sub,
                'page'      : page_num + 1,
                'char_count': len(sub),
            })

    stats = {
        'n_pages' : n_pages,
        'n_chunks': len(chunks),
        'avg_chars': sum(c['char_count'] for c in chunks) // max(len(chunks), 1),
    }
    return chunks, stats


t0 = time.time()
chunks, stats = load_pdf_pypdf_layout(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pypdf layout extraction : {elapsed:.0f}ms")
print(f"Pages                   : {stats['n_pages']}")
print(f"Chunks                  : {stats['n_chunks']}")
print(f"Avg chunk length        : {stats['avg_chars']} chars")


## Step 6 — Index

In [ ]:
print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in chunks], label='[index]')

pts = [
    PointStruct(
        id=str(uuid.uuid4()),
        vector=embs[i],
        payload={
            'text'    : chunks[i]['text'],
            'metadata': {
                'page'      : chunks[i]['page'],
                'char_count': chunks[i]['char_count'],
                'source'    : 'climate.pdf',
                'pdf_lib'   : 'pypdf-layout',
            }
        }
    )
    for i in range(len(chunks))
]
qdrant.upsert(collection_name=COLLECTION_NAME, points=pts)
print(f"Indexed {qdrant.get_collection(COLLECTION_NAME).points_count} vectors in {time.time()-t0:.1f}s")


## Step 7 — Streaming LLM via `invoke_model_with_response_stream`

Bedrock's streaming API returns an `EventStream`. Each event in the stream contains
a `chunk` with a `bytes` payload that is a partial JSON object. For Anthropic models
the relevant event types are:

| Event type | Content |
|-----------|--------|
| `content_block_delta` | `delta.text` — the next token(s) |
| `message_delta` | stop reason, output token count |
| `message_stop` | End of stream |

We measure **TTFT** (time to first token) and **total latency** separately.


In [ ]:
def stream_llm(system: str, user_content: str,
               max_tokens: int = 1024) -> Generator[str, None, Dict]:
    body = json.dumps({
        "anthropic_version": "bedrock-2023-05-31",
        "max_tokens": max_tokens,
        "system": system,
        "messages": [{"role": "user", "content": user_content}],
    })
    resp = bedrock_rt.invoke_model_with_response_stream(
        modelId=LLM_MODEL, body=body,
        contentType="application/json", accept="application/json")

    for event in resp["body"]:
        chunk_data = json.loads(event["chunk"]["bytes"])
        etype = chunk_data.get("type", "")
        if etype == "content_block_delta":
            delta = chunk_data.get("delta", {})
            if delta.get("type") == "text_delta":
                yield delta.get("text", "")


def call_llm_streaming(system: str, user_content: str,
                       max_tokens: int = 1024,
                       print_stream: bool = True) -> Dict:
    t_start = time.time()
    t_first = None
    tokens  = []

    for token in stream_llm(system, user_content, max_tokens):
        if t_first is None:
            t_first = time.time()
            if print_stream:
                print("[streaming] ", end="", flush=True)
        tokens.append(token)
        if print_stream:
            print(token, end="", flush=True)

    t_end = time.time()
    if print_stream:
        print()  # newline after stream

    ttft        = (t_first - t_start) * 1000 if t_first else 0
    total_ms    = (t_end   - t_start) * 1000
    full_answer = "".join(tokens)
    char_count  = len(full_answer)

    return {
        'answer'    : full_answer,
        'ttft_ms'   : ttft,
        'total_ms'  : total_ms,
        'chars'     : char_count,
        'chars_per_s': char_count / max((t_end - (t_first or t_start)), 0.001),
    }


# Smoke test
print("Smoke test — streaming a short response:")
smoke = call_llm_streaming(
    system="You are a helpful assistant.",
    user_content="Say 'streaming works' and nothing else.",
    max_tokens=20,
)
print(f"TTFT={smoke['ttft_ms']:.0f}ms  total={smoke['total_ms']:.0f}ms")


## Step 8 — Streaming RAG Pipeline

In [ ]:
SYSTEM_PROMPT = """You are a knowledgeable assistant answering questions about a climate document.
Answer ONLY from the provided context. Be concise and precise.
If the answer is not in the context, say so explicitly.
"""


def streaming_rag(question: str, print_stream: bool = True) -> Dict:
    t0 = time.time()

    # 1. Retrieve
    qvec = embed_text(question)
    resp = qdrant.query_points(
        collection_name=COLLECTION_NAME, query=qvec,
        limit=TOP_K, with_payload=True)
    hits = [{'text': p.payload['text'],
             'page': p.payload.get('metadata', {}).get('page', '?'),
             'score': p.score} for p in resp.points]

    t_retrieve = (time.time() - t0) * 1000

    # 2. Build prompt
    context  = '\n\n'.join(
        f"[{i+1}] page={h['page']} score={h['score']:.3f}\n{h['text']}"
        for i, h in enumerate(hits))
    user_msg = f"Context:\n{context}\n\nQuestion: {question}"

    if print_stream:
        print(f"Q: {question}")
        print(f"   [retrieved {len(hits)} chunks in {t_retrieve:.0f}ms]")

    # 3. Stream generation
    stream_result = call_llm_streaming(
        SYSTEM_PROMPT, user_msg, print_stream=print_stream)

    if print_stream:
        print(f"   TTFT={stream_result['ttft_ms']:.0f}ms  "
              f"total={stream_result['total_ms']:.0f}ms  "
              f"throughput={stream_result['chars_per_s']:.0f} chars/s")
        print()

    return {
        'question'   : question,
        'answer'     : stream_result['answer'],
        'n_chunks'   : len(hits),
        't_retrieve' : t_retrieve,
        'ttft_ms'    : stream_result['ttft_ms'],
        'total_ms'   : stream_result['total_ms'],
        'chars_per_s': stream_result['chars_per_s'],
    }


print("streaming_rag() defined.")


## Step 9 — Streaming Demo

In [ ]:
questions = [
    "What is Numerical Weather Prediction and how does it work?",
    "What types of satellites are used for weather observation?",
    "What are the five weather forecasting methods described in the document?",
]

results = []
print("=" * 70)
for q in questions:
    r = streaming_rag(q, print_stream=True)
    results.append(r)
    print("-" * 70)


## Step 10 — Streaming vs Blocking Latency Comparison

Run the same questions with the blocking API and compare total latency and
effective perceived latency (TTFT for streaming vs full wait for blocking).


In [ ]:
blocking_results = []
for r in results:
    qvec = embed_text(r['question'])
    resp = qdrant.query_points(
        collection_name=COLLECTION_NAME, query=qvec,
        limit=TOP_K, with_payload=True)
    hits = [p.payload['text'] for p in resp.points]
    context  = '\n\n'.join(f"[{i+1}]\n{h}" for i, h in enumerate(hits))
    user_msg = f"Context:\n{context}\n\nQuestion: {r['question']}"

    t0 = time.time()
    _  = call_llm_blocking(SYSTEM_PROMPT, user_msg)
    blocking_ms = (time.time() - t0) * 1000
    blocking_results.append(blocking_ms)

print(f"{'Question':<45} {'Stream TTFT':>12} {'Stream total':>13} {'Blocking total':>15}")
print("-" * 88)
for r, blk in zip(results, blocking_results):
    print(f"{r['question'][:44]:<45} "
          f"{r['ttft_ms']:>11.0f}ms "
          f"{r['total_ms']:>12.0f}ms "
          f"{blk:>14.0f}ms")

avg_ttft    = sum(r['ttft_ms']  for r in results) / len(results)
avg_stream  = sum(r['total_ms'] for r in results) / len(results)
avg_block   = sum(blocking_results) / len(blocking_results)
print()
print(f"Average TTFT (streaming first token) : {avg_ttft:.0f}ms")
print(f"Average total (streaming)            : {avg_stream:.0f}ms")
print(f"Average total (blocking)             : {avg_block:.0f}ms")
print(f"Perceived latency reduction          : {avg_block - avg_ttft:.0f}ms ({(avg_block-avg_ttft)/avg_block*100:.0f}%)")


## Step 11 — Summary

### Streaming RAG at a glance

| Metric | Blocking | Streaming |
|--------|----------|----------|
| User sees first token | After full generation | Within TTFT (~500ms) |
| Total latency | Same | Same (slightly higher overhead) |
| Perceived latency | = Total latency | = TTFT |
| Implementation | `invoke_model()` | `invoke_model_with_response_stream()` |
| Event type to parse | Single JSON | `content_block_delta` events |

### Stream event parsing

```python
for event in resp["body"]:
    data  = json.loads(event["chunk"]["bytes"])
    if data["type"] == "content_block_delta":
        token = data["delta"]["text"]   # yield this
```

### pypdf extraction modes

| Mode | Whitespace | Best for |
|------|-----------|----------|
| `extract_text()` | Minimal | Simple single-column PDFs |
| `extract_text(extraction_mode="layout")` | Preserves horizontal spacing | Multi-column, table-heavy PDFs |

> **Next: [28 — Caching RAG](28_Caching_RAG.ipynb)** —
> cache embeddings and retrieval results to cut latency and cost on repeated queries.


In [ ]:
print(f"Collection '{COLLECTION_NAME}': {qdrant.get_collection(COLLECTION_NAME).points_count} vectors")
print(f"PDF framework  : pypdf extract_text(extraction_mode='layout')")
print(f"RAG pattern    : Streaming RAG — invoke_model_with_response_stream, TTFT tracking")
print(f"Avg TTFT       : {avg_ttft:.0f}ms  vs blocking total: {avg_block:.0f}ms")
print()
print("Notebook 27 — Streaming RAG complete.")
